Import necessary libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
from scipy import stats

%matplotlib inline

# EDA 

Explore the datasets- the employees data consists of 8 columns and the demographic consists of 13 columns. \
The goal of the project is to answer following questions; \
1] Who are the top ten people with the highest salary? \
2] Which state has the highest collective salary? \
3] Which state has the most employees working at MegaCorp?

In [ ]:
url = 'https://hiring-on-live-at-shareworks.s3.amazonaws.com/employee_data.csv.bz2'
url2 = 'https://hiring-on-live-at-shareworks.s3.amazonaws.com/demographic_data.csv.bz2'
employee = pd.read_csv(url)
demographic = pd.read_csv(url2)

In [ ]:
employee.head(2)

In [ ]:
demographic.head(2)

In [ ]:
print('Size of the employee and demographic data respectively: {}, {}'.format(employee.shape, demographic.shape))

We can combine the two datasets on the 'Employee number' which is common to 
both datasets. Once merged it is easy to analyse the data further.

In [ ]:
df = pd.merge(employee, demographic, on='Employee Number')
df.head()

In [ ]:
print('Size of the entire data: {}'.format(df.shape))

Let's start with some EDA. We will first look at how much of the data is missing. Since the focus here is on the Employee['Salary'] column, we can fill in any missing data with the median. Although the salary depends on the job title and experience level, at this point we do not have either of these informations. The median would therefore be a best option to fill in missing data, as the outliers don't affect median as much as it affects mean. 

In [ ]:
df['Salary'].isnull().sum()

We seem to have no missing data, which is good! Let's now check some basic stats on the salary column.

In [ ]:
pd.options.display.float_format = '{:.2f}'.format
df['Salary'].describe()

The mean salary of the employees is around 55k with the standard deviation of about 33k. Some employees seems to have very high salary as can be noted from max, although mean is still around 55k. We also have minimum salary of zero.  We can visualize this information.

In [ ]:
sns.set(style="whitegrid")
ax = sns.boxplot(x=df["Salary"])

In [ ]:
sns.distplot(df.Salary, hist=False) #this plot shows the distribution of the salary
plt.title('Salary distrubution')
plt.ylabel('Density')

In [ ]:
# In this plot, I fit the probability density function using the estimated parameters 
# mean and S.D from the data. Compare it to the histogram and we see that it is a descent match. 
mu = 5.516132e+04  # mean of distribution
sigma = 3.281653e+04  # standard deviation of distribution

num_bins = 20

fig, ax = plt.subplots()

# the histogram of the data
n, bins, patches = ax.hist(employee['Salary'], num_bins, density=1)

# add a 'best fit' line
y = ((1 / (np.sqrt(2 * np.pi) * sigma)) *
     np.exp(-0.5 * (1 / sigma * (bins - mu))**2))
ax.plot(bins, y, '--')
ax.set_xlabel('employee_salary')
ax.set_ylabel('Probability density')
ax.set_title(r'Histogram of IQ: $\mu=55161.32$, $\sigma=32816.53$')

# Tweak spacing to prevent clipping of ylabel
fig.tight_layout()
plt.show()

In [ ]:
stats.mode(df.Salary)

Firstly, we see that there's no missing data in the Salary column. Secondly, the boxplot revels that there's lots of outliers. About 232k employees with zero salary. Definitely need further inspection to make sure they are really outliers. The histogram also shows that an abundance of people gets paid  ~ 54k. The employee income varies from 0 to  230k.\


Let's now investigate why we have employees with zero salary, the age distribution of the employees and how long the employees have been working at the company. In order to do these, first I convert the types of ['Date of Birth', 'Employee start date'] columns to datetime.

In [ ]:
df.dtypes

In [ ]:
df['Employee Start Date'] = pd.to_datetime(df['Employee Start Date'])
df['Date of Birth'] = pd.to_datetime(df['Date of Birth'])

In [ ]:
df.dtypes

In [ ]:
print(df[['Employee Start Date', 'Date of Birth']].min())
print('\n')
print(df[['Employee Start Date', 'Date of Birth']].max())


Here we see that we have employees who joined the company in 1960's and are still at the company as well as the employees who recently joined the company. We can also see that the most recent employees who joined the company will be starting work at the end of the year, so that explains why we have lots of employees with zero salaries. 

I will now add few new columns to the dataset, one will be the age of the employees column and the other will be years spent at the company.

In [ ]:
import datetime
from datetime import date

def calculateAge(birthDate):
    today = date.today()
    age = today.year - birthDate.year - \
        ((today.month, today.day) < (birthDate.month, birthDate.day))
    return age

In [ ]:
Age = []
for age in df['Date of Birth']:
    Age.append(calculateAge(age))
df['Employee Age'] = Age

In [ ]:
Age = []
for age in df['Employee Start Date']:
    Age.append(calculateAge(age))
df['Years at Company'] = Age

In [ ]:
df.head()

In [ ]:
print('Average Number of Years at the company for currently active employees: {:.2f} years'.format(df['Years at Company'].mean()))

In [ ]:
print("Number of Years at the company varies from {} to {} years.".format(df['Years at Company'].min(), df['Years at Company'].max()))

We see that the average is definitely low given the number of years spent at the company varies from 0 to 52. This is because there's lot of employees who recently joined the company. We can also note this from the distribution below.

In [ ]:
sns.distplot(df['Years at Company'],hist=False)
plt.title('Years at Company')
plt.ylabel('Density')

In [ ]:
print('Current employees: average age = {:.1f} years old and standard deviation = {:.1f}'.format(
    df['Employee Age'].mean(), df['Employee Age'].std()))

In [ ]:
sns.distplot(df['Employee Age'],hist=False)
plt.title('Age Distrubiton')
plt.ylabel('density')

In [ ]:
df['Employee Age'].describe()

Interestingly the company have employees as young as 14yrs old. Having done some exploration, I will answer the questions below.

# Who are the top ten people with the highest salary? 

We can use python's builtin function .nlargest to get the largest n elements. Here's I have limited the columns to just Full name, state and salary. 

In [ ]:
df.nlargest(10,['Salary'])[['First Name', 'Last Name', 'Middle Name', 'Salary','Home Address: State/Province']]

# Which state has the highest collective salary? 

In order to find the highest collective salary of any state, we can first group the data by state and then we can compute the total income per state. This can be easily done in 2 ways. \
Although we just want the highest collective salary of one state, I have shown the first 5 states for illustrative purpose. 

In [ ]:
collective_state_salary= df.groupby('Home Address: State/Province').agg({'Salary':['sum']})[:5]
collective_state_salary.columns = ['Total Income per state']
collective_state_salary = collective_state_salary.reset_index()
collective_state_salary

# Which state has the most employees working at MegaCorp?

In [ ]:
pd.DataFrame(df['Home Address: State/Province'].value_counts())[:5]

So 'Arizona' has most employees working at MegaCorp. Again, for illustrative purpose I show the top 5. 

In [ ]:
df.isna().sum()# just to make sure we don't have any missing info in other columns.